In [1]:
import json
import sys

import httpx

API_URL = "http://localhost:7100"

In [2]:
query = "medical graph rag"
payload = {
    "query": query,
    "num_topics": 1, # fan-out only 1 for visibility
    "max_attempts": 2,
    "top_k": 5,
}

In [3]:
def handle_event(event_type: str | None, data: dict) -> None:
    if event_type == "node_start":
        node = data.get("node", "?")
        topic = data.get("topic")
        label = f"  [{topic}]" if topic else ""
        print(f"▶ {node}{label}")

    elif event_type == "node_end":
        node = data.get("node", "?")
        topic = data.get("topic")
        detail = _end_detail(node, data)
        label = f"  [{topic}]" if topic else ""
        print(f"✓ {node}{label}  {detail}")

    elif event_type == "done":
        topics = data.get("topics", [])
        papers = data.get("papers", [])
        print()
        print(f"══ Done ══  topics={topics}")
        for i, p in enumerate(papers, 1):
            score = p.get("relevance_score", 0)
            title = p.get("title", "Untitled")
            print(f"  {i}. [{score:.2f}] {title}")

    elif event_type == "error":
        print(f"✗ ERROR: {data.get('error', data)}")

    else:
        print(f"  ({event_type}) {json.dumps(data)[:120]}")


def _end_detail(node: str, data: dict) -> str:
    if node == "generate_topics":
        return f"topics={data.get('topics', [])}"
    if node == "generate_query":
        return f"query={data.get('search_query', '')!r}"
    if node == "search":
        return f"papers_found={data.get('papers_found', 0)}"
    if node == "judge":
        return f"papers_relevant={data.get('papers_relevant', 0)}"
    if node == "generate_feedback":
        fb = (data.get("feedback") or "")[:80]
        return f"feedback={fb!r}"
    if node == "collect_and_rank":
        return f"papers={data.get('papers_count', 0)}"
    if node == "search_subgraph":
        return f"emitted={data.get('papers_emitted', 0)}"
    return ""

In [4]:
# Stream
with httpx.stream(
    "POST",
    f"{API_URL}/search/stream",
    json=payload,
    timeout=120.0,
) as resp:
    resp.raise_for_status()

    event_type = None
    for line in resp.iter_lines():
        if line.startswith("event: "):
            event_type = line[7:].strip()
        elif line.startswith("data: "):
            data = json.loads(line[6:])
            handle_event(event_type, data)
        elif line == "":
            event_type = None

▶ generate_topics
✓ generate_topics  topics=['medical graph retrieval and generation for clinical decision support']
▶ search_subgraph  [medical graph retrieval and generation for clinical decision support]
▶ generate_query  [medical graph retrieval and generation for clinical decision support]
✓ generate_query  [medical graph retrieval and generation for clinical decision support]  query='medical graph retrieval generation clinical decision support'
▶ search  [medical graph retrieval and generation for clinical decision support]
✓ search  [medical graph retrieval and generation for clinical decision support]  papers_found=5
▶ judge  [medical graph retrieval and generation for clinical decision support]
✓ judge  [medical graph retrieval and generation for clinical decision support]  papers_relevant=4
▶ emit  [medical graph retrieval and generation for clinical decision support]
✓ emit  [medical graph retrieval and generation for clinical decision support]  
✓ search_subgraph  [medical 